# Bronze_BIX_LAH_DF2

RealPage SFTP → **Heritage_Bronze_LH** (raw CSVs only).

SCD2 + computed columns se ejecutan en **Silver_BIX_LAH_DF2**.

| Step | Descripcion |
|---|---|
| **Step 1** | SFTP bimft.realpage.com → Files/RealPageDaily |

In [ ]:
# %pip install paramiko

## Tables Configuration

17 BIX tables with business keys and hash column exclusions.

In [ ]:
TECH_SCD2_COLS = ["hash_value", "Start_Date", "End_Date", "IsCurrent"]

tables = [
  {
    "name": "dimaccountgrouphierarchy__0001",
    "type": "SCD2",
    "business_key": ["AccountGroupHierarchyKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]
  },
  {
    "name": "dimclassificationlist__0001",
    "type": "SCD2",
    "business_key": ["ClassificationListKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]
  },
  {
    "name": "dimleadmanagement__0001",
    "type": "SCD2",
    "business_key": ["LeadManagementKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]
  },
  {
    "name": "dimleadmetrics__0001",
    "type": "SCD2",
    "business_key": ["LeadMetricsKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate","Date","Processed_Time",*TECH_SCD2_COLS]
  },
  {
    "name": "dimleaseattributes__0001",
    "type": "SCD2",
    "business_key": ["LeaseAttributesKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate","CDSExtractDate",*TECH_SCD2_COLS]
  },
  {
    "name": "dimresidentactivitylog__0001",
    "type": "SCD2",
    "business_key": ["ResidentActivityLogkey"],
    "include_hash_cols": ["ResidentActivityLogkey","PropertyKey","ResidentKey","CodeActivityTypeCode"]
  },
  {
    "name": "dimleaseexpiration__0001",
    "type": "SCD2",
    "business_key": ["LeaseExpirationKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]
  },
  {
    "name": "dimmakereadyrequest__0001",
    "type": "SCD2",
    "business_key": ["MakeReadyRequestKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate","osl_ExtractDate","CreateDate","UnitKey",*TECH_SCD2_COLS]
  },
  {
    "name": "dimproperty__0001",
    "type": "SCD2",
    "business_key": ["PropertyKey"],
    "include_hash_cols": ["PropertyKey","OrganizationKey","PropertyName","PropertyAddress1"]
  },
  {
    "name": "dimresidentmember__0001",
    "type": "SCD2",
    "business_key": ["ResidentMemberKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate","CDSExtractDate",*TECH_SCD2_COLS]
  },
  {
    "name": "dimscreeningapplicant__0001",
    "type": "SCD2",
    "business_key": ["ApplicantKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate","RowStartDate","BirthDate","ApplicationDate",*TECH_SCD2_COLS]
  },
  {
    "name": "dimservicerequest__0001",
    "type": "SCD2",
    "business_key": ["ServiceRequestKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate","RequestTime","CompleteTime","UnitKey",
                          "WorkingDays","mrrrMakeReadyStartDay","MRWorkingDays","mrrrMoveOutDate",*TECH_SCD2_COLS]
  },
  {
    "name": "factoperationalkpi__0001",
    "type": "SCD2",
    "business_key": ["PropertyKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]
  },
  {
    "name": "factaccountgrouphierarchydetail__0001",
    "type": "SCD2",
    "business_key": ["AccountGroupHierarchyDetailKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]
  },
  {
    "name": "factglsummary__0001",
    "type": "SCD2",
    "business_key": ["Tablekey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]
  },
  {
    "name": "factpropertystatisticsasofdate__0001",
    "type": "SCD2",
    "business_key": ["PropertyKey","PostDate"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]
  },
  {
    "name": "factscreeningapplicant__0001",
    "type": "SCD2",
    "business_key": ["ApplicantKey"],
    "exclude_hash_cols": ["RecordCreatedDate","RecordModifiedDate",*TECH_SCD2_COLS]
  },
]

## Step 1 — SFTP Download

Downloads today's CSVs from `bimft.realpage.com:/BI_Download` into `Files/RealPageDaily`. Hard-resets the folder on every run.

In [ ]:
# ============================================================
# STEP 1 — SFTP → Lakehouse Files/RealPageDaily
# Hard reset: deletes and re-downloads all CSVs from today
# ============================================================
def step1_sftp_download():
    import os, re, stat, pathlib, zoneinfo, datetime as dt, paramiko
    import notebookutils as nbutils
    fs = nbutils.fs

    HOST, PORT = "bimft.realpage.com", 22
    USER       = "EXT-HERITAGEHILLPROPMGMT_BI"
    PWD        = "BJEEod^ykvA9Rm"
    REMOTE_DIR = "/BI_Download"
    TMP_DIR    = pathlib.Path("/tmp/realpage")
    LAKE_DIR   = "Files/RealPageDaily"

    tz    = zoneinfo.ZoneInfo("America/New_York")
    token = dt.datetime.now(tz).strftime("%Y%m%d")
    rx    = re.compile(rf"{token}.*\.csv$", re.I)
    print(f"Token: {token} | Remote: {REMOTE_DIR} | Lakehouse: {LAKE_DIR}")

    # Hard reset
    try:
        fs.rm(LAKE_DIR, True)
        print("Folder deleted:", LAKE_DIR)
    except Exception:
        print("Folder did not exist, continuing...")
    fs.mkdirs(LAKE_DIR)
    TMP_DIR.mkdir(exist_ok=True, parents=True)

    # SFTP download
    copied = []
    with paramiko.Transport((HOST, PORT)) as tr:
        tr.connect(username=USER, password=PWD)
        print("Connected to SFTP")
        with paramiko.SFTPClient.from_transport(tr) as sftp:
            try:
                entries = sftp.listdir_attr(REMOTE_DIR)
            except FileNotFoundError:
                REMOTE_DIR = REMOTE_DIR.lstrip("/")
                entries    = sftp.listdir_attr(REMOTE_DIR)
            for e in entries:
                if stat.S_ISREG(e.st_mode) and rx.search(e.filename):
                    local = TMP_DIR / e.filename
                    sftp.get(f"{REMOTE_DIR}/{e.filename}", str(local))
                    fs.fastcp(f"file:{local}", f"{LAKE_DIR}/{e.filename}", True)
                    local.unlink(missing_ok=True)
                    copied.append(e.filename)
    print(f"Copied {len(copied)} file(s)")

    # Clean names: keep only part after dbo__
    regex = re.compile(r".*dbo__(.+)", re.I)
    renamed = []
    for entry in fs.ls(LAKE_DIR):
        fname = entry.path.split("/")[-1]
        m = regex.match(fname)
        if m:
            dst = f"{LAKE_DIR}/{m.group(1)}"
            fs.mv(entry.path, dst, True, True)
            renamed.append((fname, m.group(1)))
    print(f"Renamed {len(renamed)} file(s)")
    for old, new in renamed:
        print(f"  {old} → {new}")

## Step 2 — SCD Type 2 Engine

For each table in the array: read CSV → standardize dates → compute SHA-256 hash → write `stg_<table>` → SCD2 merge into `<table>` (detect new rows, changed rows, close old versions).

In [ ]:
# ============================================================
# STEP 2 — Files/RealPageDaily → STG → SCD2 Delta Tables
# For every table in the `tables` array:
#   1. Read CSV  2. Standardize dates  3. Compute hash
#   4. Write stg_<table>  5. SCD2 merge into <table>
# ============================================================
import re
from datetime import timedelta, date
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DateType, TimestampType
from pyspark.sql.window import Window

HASH_COL      = "hash_value"
START_COL     = "Start_Date"
END_COL       = "End_Date"
ISCURRENT_COL = "IsCurrent"
MAX_END       = "12/31/9999"
DEFAULT_START = "1/1/2025"
RAW_DIR       = "Files/RealPageDaily"

spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

DATE_PATTERNS = [
    "M/d/yyyy","MM/dd/yyyy","M/d/yy","MM/dd/yy",
    "yyyy-MM-dd","yyyy/MM/dd","yyyyMMdd",
    "dd-MMM-yyyy","d-MMM-yyyy","MMM d, yyyy","MMMM d, yyyy"
]
TS_PATTERNS = [
    "M/d/yyyy h:mm:ss a","MM/dd/yyyy h:mm:ss a",
    "M/d/yyyy H:mm:ss","MM/dd/yyyy H:mm:ss",
    "M/d/yyyy HH:mm:ss","MM/dd/yyyy HH:mm:ss",
    "yyyy-MM-dd HH:mm:ss","yyyy-MM-dd'T'HH:mm:ss","yyyy-MM-dd'T'HH:mm:ss.SSS"
]

def _parsed_date_expr(col_trim):
    parsed_ts = F.coalesce(*[F.to_timestamp(col_trim, p) for p in TS_PATTERNS])
    parsed_d  = F.coalesce(*[F.to_date(col_trim, p) for p in DATE_PATTERNS])
    return F.coalesce(F.to_date(parsed_ts), parsed_d)

def standardize_date_columns(df, sample_rows=1000, success_threshold=0.90, min_non_null=10):
    out = df
    sample = df.limit(sample_rows)
    for field in df.schema.fields:
        c   = field.name
        dtp = field.dataType
        if isinstance(dtp, (DateType, TimestampType)):
            out = out.withColumn(c, F.date_format(F.to_date(F.col(c)), "M/dd/yyyy"))
            continue
        if isinstance(dtp, StringType):
            col_trim    = F.trim(F.col(c))
            parsed_date = _parsed_date_expr(col_trim)
            stats = sample.select(
                F.sum(F.when(col_trim.isNotNull() & (F.length(col_trim) > 0), 1).otherwise(0)).alias("non_null"),
                F.sum(F.when((col_trim.isNotNull() & (F.length(col_trim) > 0)) & parsed_date.isNotNull(), 1).otherwise(0)).alias("parsed_ok")
            ).collect()[0]
            non_null  = int(stats["non_null"] or 0)
            parsed_ok = int(stats["parsed_ok"] or 0)
            rate = (parsed_ok / non_null) if non_null > 0 else 0.0
            if non_null >= min_non_null and rate >= success_threshold:
                out = out.withColumn(c, F.date_format(parsed_date, "M/dd/yyyy"))
    return out

def _norm_key(s):
    return re.sub(r"[^a-z0-9]+", "", (s or "").lower().strip())

def normalize_excludes(df, exclude_cols):
    cols_map = {_norm_key(c): c for c in df.columns}
    return [cols_map[_norm_key(str(x))] for x in (exclude_cols or []) if _norm_key(str(x)) in cols_map]

def add_hash(df, exclude_cols=None, include_cols=None):
    if include_cols:
        cols_for_hash = sorted([c for c in df.columns if c in set(include_cols)], key=str.lower)
    else:
        exclude_set   = set((exclude_cols or []) + [HASH_COL, START_COL, END_COL, ISCURRENT_COL])
        cols_for_hash = sorted([c for c in df.columns if c not in exclude_set], key=str.lower)
    if not cols_for_hash:
        return df.withColumn(HASH_COL, F.sha2(F.lit(""), 256))
    exprs = [F.coalesce(F.trim(F.col(c).cast("string")), F.lit("")) for c in cols_for_hash]
    return df.withColumn(HASH_COL, F.sha2(F.concat_ws("||", *exprs), 256))

def _resolve_hash_kwargs(df, cfg):
    incl = cfg.get("include_hash_cols")
    if incl:
        return {"include_cols": normalize_excludes(df, incl)}
    return {"exclude_cols": normalize_excludes(df, cfg.get("exclude_hash_cols", []))}

def _order_ts(df, colname):
    if colname not in df.columns: return None
    return F.to_timestamp(_parsed_date_expr(F.trim(F.col(colname).cast("string"))))

def dedup_child_latest(df, bk_cols):
    order_cols   = ["RecordModifiedDate","RecordCreatedDate","CDSExtractDate","osl_ExtractDate","CreateDate","RowStartDate","ModifyDate","PostDate"]
    order_exprs  = [ts.desc_nulls_last() for c in order_cols for ts in [_order_ts(df, c)] if ts is not None]
    if HASH_COL in df.columns:    order_exprs.append(F.col(HASH_COL).desc_nulls_last())
    for c in bk_cols:
        if c in df.columns:       order_exprs.append(F.col(c).cast("string").desc_nulls_last())
    if not order_exprs:
        return df.dropDuplicates(bk_cols)
    w = Window.partitionBy(*[F.col(c) for c in bk_cols]).orderBy(*order_exprs)
    return df.withColumn("_rn", F.row_number().over(w)).where(F.col("_rn") == 1).drop("_rn")

def ensure_tech_cols(df):
    if START_COL     not in df.columns: df = df.withColumn(START_COL,     F.lit(DEFAULT_START))
    if END_COL       not in df.columns: df = df.withColumn(END_COL,       F.lit(MAX_END))
    if ISCURRENT_COL not in df.columns: df = df.withColumn(ISCURRENT_COL, F.lit(1))
    return df

def _has_data(df):
    return df.limit(1).count() > 0

def consolidate_unique_hash_ranges(df, bk_cols):
    start_p = _parsed_date_expr(F.trim(F.col(START_COL).cast("string")))
    end_p   = F.when(F.trim(F.col(END_COL)) == F.lit(MAX_END), F.to_date(F.lit(MAX_END), "M/d/yyyy"))\
               .otherwise(_parsed_date_expr(F.trim(F.col(END_COL).cast("string"))))
    grp     = bk_cols + [HASH_COL]
    w       = Window.partitionBy(*[F.col(c) for c in grp])
    wfirst  = Window.partitionBy(*[F.col(c) for c in grp]).orderBy(start_p.asc_nulls_last())
    df = df.withColumn("_sp", start_p).withColumn("_ep", end_p)
    df = df.withColumn("_smin", F.min("_sp").over(w)).withColumn("_emax", F.max("_ep").over(w))
    df = df.withColumn("_rn", F.row_number().over(wfirst)).where(F.col("_rn") == 1).drop("_rn")
    df = df.withColumn(START_COL, F.date_format("_smin", "M/dd/yyyy"))
    df = df.withColumn(END_COL,
        F.when(F.col("_emax") == F.to_date(F.lit(MAX_END), "M/d/yyyy"), F.lit(MAX_END))
         .otherwise(F.date_format("_emax", "M/dd/yyyy")))
    # Fix overlapping ends
    start2 = _parsed_date_expr(F.trim(F.col(START_COL).cast("string")))
    end2   = F.when(F.trim(F.col(END_COL)) == F.lit(MAX_END), F.to_date(F.lit(MAX_END), "M/d/yyyy"))\
              .otherwise(_parsed_date_expr(F.trim(F.col(END_COL).cast("string"))))
    wbk    = Window.partitionBy(*[F.col(c) for c in bk_cols]).orderBy(start2.asc_nulls_last())
    next_s = F.lead(start2).over(wbk)
    end_fix = F.when((F.trim(F.col(END_COL)) == F.lit(MAX_END)) & next_s.isNotNull(), F.date_sub(next_s, 1)).otherwise(end2)
    end_fix = F.greatest(start2, end_fix)
    df = df.withColumn("_ef", end_fix)
    df = df.withColumn(END_COL,
        F.when(F.col("_ef") == F.to_date(F.lit(MAX_END), "M/d/yyyy"), F.lit(MAX_END))
         .otherwise(F.date_format("_ef", "M/dd/yyyy")))
    df = df.drop("_sp","_ep","_smin","_emax","_ef")
    return df.withColumn(ISCURRENT_COL, F.when(F.trim(F.col(END_COL)) == F.lit(MAX_END), F.lit(1)).otherwise(F.lit(0)))

def force_single_current(df, bk_cols):
    end_d = F.when(F.trim(F.col(END_COL)) == F.lit(MAX_END), F.to_date(F.lit(MAX_END), "M/d/yyyy"))\
             .otherwise(_parsed_date_expr(F.trim(F.col(END_COL).cast("string"))))
    start_d = _parsed_date_expr(F.trim(F.col(START_COL).cast("string")))
    w = Window.partitionBy(*[F.col(c) for c in bk_cols]).orderBy(end_d.desc_nulls_last(), start_d.desc_nulls_last())
    df2 = df.withColumn("_rc", F.row_number().over(w))
    return df2.withColumn(END_COL, F.when(F.col("_rc")==1, F.lit(MAX_END)).otherwise(F.col(END_COL)))\
              .withColumn(ISCURRENT_COL, F.when(F.col("_rc")==1, F.lit(1)).otherwise(F.lit(0))).drop("_rc")

def _pick_csv_path(arr_name):
    want  = _norm_key(arr_name)
    items = mssparkutils.fs.ls(RAW_DIR)
    for it in items:
        fname = getattr(it,"name",None) or it.path.split("/")[-1]
        stem  = re.sub(r"\.csv$","", fname, flags=re.I)
        if _norm_key(stem) == want:
            return getattr(it,"path", f"{RAW_DIR}/{fname}")
    raise ValueError(f"No CSV for {arr_name} in {RAW_DIR}")

def _read_raw(arr_name):
    path = _pick_csv_path(arr_name)
    return spark.read.option("header",True).option("inferSchema",True).csv(path)

def _try_table(name):
    try: return spark.table(name)
    except Exception:
        db = spark.catalog.currentDatabase()
        for t in spark.catalog.listTables(db):
            if (t.name or "").lower() == name.lower():
                return spark.table(t.name)
        raise

def _load_parent_or_empty(parent, ref_schema_df):
    try:    return _try_table(parent)
    except: return spark.createDataFrame([], ref_schema_df.schema)

def scd2_update(cfg):
    name   = cfg["name"]
    bk     = cfg.get("business_key", [])
    parent = name
    child  = f"stg_{name}"
    if not bk: raise ValueError(f"{name}: empty business_key")

    print(f"  Parent: {parent} | Child: {child}")

    # RAW → STG with hash
    df_raw = _read_raw(name)
    df_raw = standardize_date_columns(df_raw)
    df_raw = ensure_tech_cols(df_raw)
    df_stg = add_hash(df_raw, **_resolve_hash_kwargs(df_raw, cfg))
    df_stg.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(child)
    print(f"  STG overwritten: {child} | rows={df_stg.count()}")

    # SCD2 merge
    df_child  = _try_table(child)
    df_parent = _load_parent_or_empty(parent, df_child)
    df_parent = standardize_date_columns(df_parent)
    df_parent = ensure_tech_cols(df_parent)
    df_parent = add_hash(df_parent, **_resolve_hash_kwargs(df_parent, cfg))

    df_child  = dedup_child_latest(df_child, bk).cache()
    _ = df_child.count()

    TODAY     = date.today()
    YESTERDAY = TODAY - timedelta(days=1)
    TODAY_STR = f"{TODAY.month}/{TODAY.day}/{TODAY.year}"
    YEST_STR  = f"{YESTERDAY.month}/{YESTERDAY.day}/{YESTERDAY.year}"

    c   = df_child.alias("c")
    pc  = df_parent.where(F.trim(F.col(END_COL)) == F.lit(MAX_END)).alias("pc")
    ph  = df_parent.where(F.trim(F.col(END_COL)) != F.lit(MAX_END)).alias("ph")
    cond_cp = [F.col(f"c.{x}").cast("string") == F.col(f"pc.{x}").cast("string") for x in bk]

    changed_keys = (c.join(pc, on=cond_cp, how="inner")
        .where(F.col(f"c.{HASH_COL}") != F.col(f"pc.{HASH_COL}"))
        .select([F.col(f"c.{x}").alias(x) for x in bk]).dropDuplicates(bk))

    new_bk = c.join(pc, on=cond_cp, how="left_anti").select("c.*").dropDuplicates(bk)

    still_ok = pc.select("pc.*")
    to_close = None
    if _has_data(changed_keys):
        k = changed_keys.alias("k")
        cond_pk = [F.col(f"pc.{x}").cast("string")==F.col(f"k.{x}").cast("string") for x in bk]
        start_p = _parsed_date_expr(F.trim(F.col(f"pc.{START_COL}").cast("string")))
        yest_p  = _parsed_date_expr(F.lit(YEST_STR))
        today_p = _parsed_date_expr(F.lit(TODAY_STR))
        safe_end = F.least(today_p, F.greatest(start_p, yest_p))
        to_close  = pc.join(k, on=cond_pk, how="inner").select("pc.*")\
                      .withColumn(END_COL, F.date_format(safe_end, "M/dd/yyyy"))
        still_ok  = pc.join(k, on=cond_pk, how="left_anti").select("pc.*")

    df_out = ph.unionByName(still_ok, allowMissingColumns=True)
    if to_close is not None:
        df_out = df_out.unionByName(to_close, allowMissingColumns=True)
    if _has_data(new_bk):
        df_out = df_out.unionByName(
            new_bk.withColumn(START_COL, F.lit(TODAY_STR)).withColumn(END_COL, F.lit(MAX_END)),
            allowMissingColumns=True)
    if _has_data(changed_keys):
        k2 = changed_keys.alias("k2")
        cond_ck = [F.col(f"c.{x}").cast("string")==F.col(f"k2.{x}").cast("string") for x in bk]
        changed_new = c.join(k2, on=cond_ck, how="inner").select("c.*").dropDuplicates(bk)
        if _has_data(changed_new):
            df_out = df_out.unionByName(
                changed_new.withColumn(START_COL, F.lit(TODAY_STR)).withColumn(END_COL, F.lit(MAX_END)),
                allowMissingColumns=True)

    df_out = standardize_date_columns(df_out)
    df_out = consolidate_unique_hash_ranges(df_out, bk)
    df_out = force_single_current(df_out, bk)
    df_out.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(parent)
    df_child.unpersist()
    print(f"  SCD2 OK: {parent}")

def step2_run_scd2():
    from notebookutils import mssparkutils as ms
    global mssparkutils
    mssparkutils = ms
    print(f"Running SCD2 for {len(tables)} tables...")
    failed = []
    for cfg in tables:
        nm = cfg.get("name","unknown")
        try:
            print(f"\n=== {nm} ===")
            scd2_update(cfg)
            print(f"DONE: {nm}")
        except Exception as e:
            failed.append((nm, str(e)))
            import traceback; traceback.print_exc()
            print(f"FAIL: {nm}: {str(e)[:400]}")
    print(f"\nSCD2 finished. OK={len(tables)-len(failed)} | FAIL={len(failed)}")
    if failed:
        print("Failed tables:", [n for n,_ in failed])
        raise Exception(f"SCD2 failed on {len(failed)} tables: {failed[0][0]}")

## Step 3 — dimservicerequest: Computed Columns

Replicates three DAX calculated columns:

| Column | Logic |
|---|---|
| `WorkingDays` | Business days from `CreateDate` → `ActualCompletedDate` (or TODAY if blank) using `Date[NonWorkDays]` |
| `mrrrMoveOutDate` | `LOOKUPVALUE(dimmakereadyrequest[mrrMoveOutDate])` when `ServiceRequestTypeCode = MR` |
| `MRWorkingDays` | Business days from `mrrrMoveOutDate` → `ActualCompletedDate` for MR rows |

In [ ]:
# ============================================================
# STEP 3 — dimservicerequest__0001: Computed Columns
#
# Replicates DAX calculated columns:
#   WorkingDays      = business days CreateDate → ActualCompletedDate
#   mrrMoveOutDate   = LOOKUPVALUE(dimmakereadyrequest, key) when TypeCode="MR"
#   MRWorkingDays    = business days mrrMoveOutDate → ActualCompletedDate (MR only)
# Uses: Date table (NonWorkDays=0 → workday), dimmakereadyrequest__0001
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def _parse_dt(col_):
    # Parse date from M/d/yyyy, yyyy-MM-dd or timestamp strings
    s = F.trim(col_.cast("string"))
    s_norm = F.when(
        s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"),
        F.concat_ws("/",
            F.lpad(F.split(s, "/").getItem(0), 2, "0"),
            F.lpad(F.split(s, "/").getItem(1), 2, "0"),
            F.split(s, "/").getItem(2))
    ).otherwise(s)
    return F.coalesce(
        F.to_date(s, "yyyy-MM-dd"),
        F.to_date(s, "yyyy-MM-dd HH:mm:ss"),
        F.to_date(s, "yyyy-MM-dd'T'HH:mm:ss"),
        F.to_date(s_norm, "MM/dd/yyyy"),
        F.to_date(s_norm, "MM/dd/yyyy HH:mm:ss"),
    )

def _build_calendar(df_date, date_col="Date", nonwork_col="NonWorkDays"):
    # Build cumulative workday calendar from Date table
    cal = (df_date
        .select(F.col(date_col).cast("date").alias("CalDate"),
                F.col(nonwork_col).cast("int").alias("NonWorkDays"))
        .filter(F.col("CalDate").isNotNull())
        .dropDuplicates(["CalDate"])
        .withColumn("IsWorkDay", F.when(F.col("NonWorkDays")==0, F.lit(1)).otherwise(F.lit(0))))
    w = Window.orderBy("CalDate").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    cal = (cal
        .withColumn("CumWorkDays",    F.sum("IsWorkDay").over(w))
        .withColumn("WorkDaysBefore", F.col("CumWorkDays") - F.col("IsWorkDay"))
        .select("CalDate", "CumWorkDays", "WorkDaysBefore"))
    cal_s = cal.select(F.col("CalDate").alias("_SC"), F.col("WorkDaysBefore").alias("_BS"))
    cal_e = cal.select(F.col("CalDate").alias("_EC"), F.col("CumWorkDays").alias("_CE"))
    return cal_s, cal_e

def _workday_diff(df, start_col, end_col, out_col, cal_s, cal_e, s_alias, e_alias):
    # Join calendar twice to compute business days between two date columns
    cs = cal_s.select(F.col("_SC").alias(s_alias+"_cal"), F.col("_BS").alias(s_alias+"_bs"))
    ce = cal_e.select(F.col("_EC").alias(e_alias+"_cal"), F.col("_CE").alias(e_alias+"_ce"))
    df = (df
        .join(cs, df[start_col] == cs[s_alias+"_cal"], "left")
        .join(ce, df[end_col]   == ce[e_alias+"_cal"], "left")
        .withColumn(out_col,
            F.when(
                F.col(start_col).isNull() | F.col(end_col).isNull() |
                F.col(s_alias+"_bs").isNull() | F.col(e_alias+"_ce").isNull() |
                (F.col(end_col) < F.col(start_col)),
                F.lit(None).cast("int")
            ).otherwise(
                F.greatest((F.col(e_alias+"_ce") - F.col(s_alias+"_bs")) - F.lit(1), F.lit(0)).cast("int")
            ))
        .drop(s_alias+"_cal", s_alias+"_bs", e_alias+"_cal", e_alias+"_ce"))
    return df

def step3_add_computed_cols():
    sr   = spark.table("dimservicerequest__0001")
    date = spark.table("Date")
    mr   = (spark.table("dimmakereadyrequest__0001")
               .select("ServiceRequestKey",
                       F.col("mrrMakeReadyStartDay").alias("_mrStartDay"),
                       F.col("mrrMoveOutDate").alias("_mrMoveOut"))
               .dropDuplicates(["ServiceRequestKey"]))

    cal_s, cal_e = _build_calendar(date)

    # Resolve end date: ActualCompletedDate or TODAY()
    sr = sr.withColumn("_EndDate",   F.coalesce(_parse_dt(F.col("ActualCompletedDate")), F.current_date()))
    sr = sr.withColumn("_StartDate", _parse_dt(F.col("CreateDate")))

    # WorkingDays: CreateDate → _EndDate
    sr = _workday_diff(sr, "_StartDate", "_EndDate", "WorkingDays", cal_s, cal_e, "wd_s", "wd_e")

    # mrrMoveOutDate + mrrrMakeReadyStartDay from dimmakereadyrequest (only for MR type)
    sr = (sr.join(mr, on="ServiceRequestKey", how="left")
            .withColumn("mrrrMakeReadyStartDay",
                F.when(F.upper(F.trim(F.col("ServiceRequestTypeCode"))) == F.lit("MR"), F.col("_mrStartDay"))
                 .otherwise(F.lit(None)))
            .withColumn("mrrrMoveOutDate",
                F.when(F.upper(F.trim(F.col("ServiceRequestTypeCode"))) == F.lit("MR"), F.col("_mrMoveOut"))
                 .otherwise(F.lit(None)))
            .drop("_mrStartDay", "_mrMoveOut"))

    # MRWorkingDays: mrrrMoveOutDate → _EndDate (MR rows only)
    sr = sr.withColumn("_MRStart", _parse_dt(F.col("mrrrMoveOutDate")))
    sr = _workday_diff(sr, "_MRStart", "_EndDate", "MRWorkingDays", cal_s, cal_e, "mr_s", "mr_e")
    sr = sr.drop("_StartDate", "_EndDate", "_MRStart")

    row_count = sr.count()
    sr.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("dimservicerequest__0001")
    print(f"step3 OK: dimservicerequest__0001 | rows={row_count}")
    sr.select("ServiceRequestKey","ServiceRequestTypeCode",
              "WorkingDays","mrrrMakeReadyStartDay","MRWorkingDays","mrrrMoveOutDate").show(10, truncate=False)

## Pipeline Runner

Executes Steps 1 → 2 → 3 in sequence. Fails the notebook if any step errors.

In [ ]:
# ============================================================
# PIPELINE RUNNER — Bronze BIX (SFTP extraction only)
# SCD2 and computed columns run in Silver_BIX_LAH_DF2
# ============================================================
import traceback

_RESULTS = []

def _run(name, fn):
    print()
    print('='*50)
    print(f"START: {name}")
    print('='*50)
    try:
        fn()
        _RESULTS.append((name, "OK"))
        print(f"OK: {name}")
    except Exception as e:
        _RESULTS.append((name, str(e)[:300]))
        print(f"FAIL: {name}")
        print(traceback.format_exc())

_run("Step 1: SFTP Download", step1_sftp_download)

print("" + "="*50)
print("BRONZE BIX PIPELINE SUMMARY")
print("="*50)
for name, status in _RESULTS:
    icon = "OK" if status == "OK" else "FAIL"
    print(f"  [{icon}] {name}" + (f": {status}" if status != "OK" else ""))

if any(s != "OK" for _, s in _RESULTS):
    raise Exception("Bronze BIX pipeline had failures.")
print("Bronze BIX extraction complete. Run Silver_BIX_LAH_DF2 next.")